# Station census notebook
Given a rectangular bounding box and an analysis period, the notebook queries the OpenAQ database for stations in
that geographical region which measure PM2.5 and whose active measurement range is larger than or equal to the analysis period

In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
# External libraries imports

import os, dotenv
from pathlib import Path
from datetime import datetime

from openaq import OpenAQ
import pandas as pd
import geopandas as gpd
from numpy import argmax

In [17]:
# Custom libraries imports

import monitoring_deserts.data.fetch_openaq as fetch_openaq


In [18]:
## Notebook level constants - later change to read from config file

# Coordinates for South Asia - India, Pakistan, Nepal, Bangladesh, Sri-Lanka + Afghanistan 
BOUNDING_BOX = BB_Xm, BB_Ym, BB_XM, BB_Ym = (60.424805,5.353521,93.383789,38.272689)
PERIOD = DAY_START, DAY_END = ["2024-01-01", "2026-01-01"] # Half-open interval

In [19]:
# Notebook starts here

dotenv.load_dotenv()
with OpenAQ() as client:
    pm25_id = fetch_openaq.get_pollutant_id(client, poll_name="PM25")
    stations_list = fetch_openaq.pull_stations(client, BOUNDING_BOX, pollutant_id=pm25_id)
    
print(f"The parameter ID for PM2.5 is {pm25_id}")
print(f"Total number of stations measuring PM2.5 in the region: {len(stations_list)}")

The parameter ID for PM2.5 is 2
Total number of stations measuring PM2.5: 1347


In [ ]:
# Dump old stuff
dotenv.load_dotenv()




with OpenAQ() as oaq_client:
    pollutant_list = oaq_client.parameters.list(parameter_type="pollutant").dict()["results"]
    pm25_oaq_id = next(x["id"] for x in pollutant_list if x["name"].upper() == "PM25")
    stations_list = oaq_client.locations.list(parameters_id=pm25_oaq_id,
                                              bbox=delhi_bbox,
                                              limit=100)
    # Select only the sensors which measure PM2.5 
    pm25_sensors_id_list = [next(sensor.id for sensor in station.sensors if sensor.parameter.id == pm25_oaq_id)  for station in stations_list.results]
    # Now pull each sensor data to check when the last measurement was
    pm25_active_sensors = []
    for i, s_id in enumerate(pm25_sensors_id_list):
        sens_result = oaq_client.sensors.get(sensors_id=s_id)
        if check_last_measurement(sens_result.results[0]):
            pm25_active_sensors.append(s_id)
    # Loop to request sensor measurements. Make a first temporary DataFrame with a station-month MultiIndex 
    df_list = []
    for s_id in pm25_active_sensors:
        meas_list = oaq_client.measurements.list(sensors_id=s_id,
                                                data="days",
                                                rollup="monthly",
                                                date_from="2024-01-01",
                                                date_to="2025-12-31",
                                                limit=30)
        df_list.append(make_df(meas_list, idx=s_id))
